In [ ]:
import json
import pandas as pd

# 1. Load the Paper Texts (Titles & Abstracts)
paper_data = []
with open("CSFCube/abstracts-csfcube-preds.jsonl", "r") as f:
    for line in f:
        paper_data.append(json.loads(line))

# Convert to a DataFrame for easy viewing and manipulation
df_papers = pd.DataFrame(paper_data)
print(f"Total papers loaded: {len(df_papers)}")

In [ ]:
with open("CSFCube/test-pid2anns-csfcube-method.json", "r") as f:
    method_ground_truths = json.load(f)

In [ ]:
first_query_id = list(method_ground_truths.keys())[0]
candidates_and_scores = method_ground_truths[first_query_id]

print(f"\nQuery Paper ID: {first_query_id}")
print(f"Number of graded candidates for this query: {len(candidates_and_scores)}")

In [ ]:
query_text = df_papers[df_papers['paper_id'] == first_query_id]['abstract'].values[0]
print(f"\nQuery Abstract Snippet: {query_text[:200]}...")

In [ ]:
import os
import re
import json
import time
import codecs
import requests

# ============================================================
# CONFIG
# ============================================================
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
# Optional: set explicitly if you want; otherwise auto-detected from /v1/models
MODEL_ID = None

FACETS = ["background", "method", "result"]
MAX_TOKENS = 4096
TEMPERATURE = 0.0
TOP_P = 1.0

# Full-list run (no sliding window)
OUTPUT_DIR = "CSFCube/eval_scripts/runs_gemma_3_12b_finetune_vllm_full_list_4096"
EXPERIMENT_NAME = "gemma_3_12b_finetune_vllm_full_list-4096"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================
def get_model_id(base_url: str) -> str:
    """Auto-detect model id from vLLM OpenAI-compatible server."""
    resp = requests.get(f"{base_url}/models", timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if "data" not in data or len(data["data"]) == 0:
        raise RuntimeError("No models returned by vLLM /v1/models.")
    return data["data"][0]["id"]

def chat_complete(base_url: str, model_id: str, user_msg: str, max_retries: int = 5) -> str:
    payload = {
        "model": model_id,
        "messages": [{"role": "user", "content": user_msg}],
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
        "stream": False,
    }

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(
                f"{base_url}/chat/completions",
                headers={"Content-Type": "application/json"},
                data=json.dumps(payload),
                timeout=180,
            )
            resp.raise_for_status()
            out = resp.json()
            return out["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            sleep_s = min(2 ** attempt, 20)
            print(f"[retry {attempt}/{max_retries}] completion failed: {e}; sleeping {sleep_s}s")
            time.sleep(sleep_s)

    raise RuntimeError(f"chat completion failed after retries: {last_err}")

def build_rerank_prompt(query_pid, cand_pids, pid2abstract):
    query_rec = pid2abstract[query_pid]
    query_text = f"Title: {query_rec['title']}\nAbstract: {query_rec['abstract']}"

    passages = []
    for i, cpid in enumerate(cand_pids):
        crec = pid2abstract[cpid]
        doc_text = " ".join(f"Title: {crec['title']}\nAbstract: {crec['abstract']}".split())
        passages.append(f"[{i+1}] {doc_text}")

    passages_str = "\n\n---\n\n".join(passages)
    n = len(cand_pids)

    return (
        f"You are an expert academic paper reranker. "
        f"Your task is to re-order the given list of passages (from [1] to [{n}]) "
        f"based on their relevance to the query. Respond with only the ranking and nothing else.\n\n"
        f"Example output for 8 passages:\n"
        f"[2] > [5] > [4] > [8] > [6] > [1] > [3] > [7]\n\n"
        f"Query: {query_text}\n\n"
        f"Passages:\n{passages_str}\n\n"
        f"Your ranking (most to least relevant):"
    )

def parse_ranking(text, n_candidates):
    indices = [int(x) for x in re.findall(r"\[(\d+)\]", text)]
    seen = set()
    deduped = []
    for idx in indices:
        if idx not in seen and 1 <= idx <= n_candidates:
            seen.add(idx)
            deduped.append(idx)

    # Fill missing positions to keep a full permutation
    for i in range(1, n_candidates + 1):
        if i not in seen:
            deduped.append(i)
    return deduped

def rank_window(query_pid, window_pids, pid2abstract, base_url, model_id):
    user_msg = build_rerank_prompt(query_pid, window_pids, pid2abstract)
    generated = chat_complete(base_url, model_id, user_msg)
    ranked_indices = parse_ranking(generated, len(window_pids))
    return [window_pids[idx - 1] for idx in ranked_indices]

# ============================================================
# LOAD DATA
# ============================================================
pid2abstract = {}
with codecs.open("CSFCube/abstracts-csfcube-preds.jsonl", "r", "utf-8") as f:
    for line in f:
        rec = json.loads(line.strip())
        pid2abstract[rec["paper_id"]] = rec

if MODEL_ID is None:
    MODEL_ID = get_model_id(VLLM_BASE_URL)
print(f"Using model id from vLLM: {MODEL_ID}")

# ============================================================
# GENERATE RANKED FILES PER FACET (FULL LIST, NO SLIDING WINDOW)
# ============================================================
for facet in FACETS:
    gold_path = f"CSFCube/test-pid2anns-csfcube-{facet}.json"
    with codecs.open(gold_path, "r", "utf-8") as fp:
        qpid2pool = json.load(fp)

    qpid2ranked = {}
    total = len(qpid2pool)

    for i, (qpid, pool_data) in enumerate(qpid2pool.items(), start=1):
        cand_pids = pool_data["cands"]
        print(f"[{facet}] {i}/{total}: {qpid} ({len(cand_pids)} candidates)")

        ranked_pids = rank_window(
            query_pid=qpid,
            window_pids=cand_pids,   # full candidate list
            pid2abstract=pid2abstract,
            base_url=VLLM_BASE_URL,
            model_id=MODEL_ID,
        )

        qpid2ranked[qpid] = [[cpid, rank] for rank, cpid in enumerate(ranked_pids)]

    out_fname = f"{OUTPUT_DIR}/test-pid2pool-csfcube-{EXPERIMENT_NAME}-{facet}-ranked.json"
    with codecs.open(out_fname, "w", "utf-8") as fp:
        json.dump(qpid2ranked, fp)
    print(f"Wrote: {out_fname}")

print("Ranking generation complete.")

# ============================================================
# OPTIONAL: run official CSFCube eval (shell commands)
# ============================================================
# python CSFCube/eval_scripts/ranking_eval.py eval_pool_ranking --gold_path CSFCube --ranked_path CSFCube/eval_scripts/runs_gemma_3_12b_finetune_vllm_full_list_4096 --experiment gemma_3_12b_finetune_vllm_full_list-4096 --facet background
# python CSFCube/eval_scripts/ranking_eval.py eval_pool_ranking --gold_path CSFCube --ranked_path CSFCube/eval_scripts/runs_gemma3_12b_finetune_vllm_full_list --experiment gemma3-12b-finetune-vllm-full-list --facet method
# python CSFCube/eval_scripts/ranking_eval.py eval_pool_ranking --gold_path CSFCube --ranked_path CSFCube/eval_scripts/runs_gemma3_12b_finetune_vllm_full_list --experiment gemma3-12b-finetune-vllm-full-list --facet result
# python CSFCube/eval_scripts/ranking_eval.py eval_pool_ranking --gold_path CSFCube --ranked_path CSFCube/eval_scripts/runs_gemma3_12b_finetune_vllm_full_list --experiment gemma3-12b-finetune-vllm-full-list --facet all